# 实验二 · 矩阵-向量乘 GEMV  (y = A · x)

**所属**：《并行计算》第三章 · ARM NEON SIMD 编程　|　**难度**：⭐⭐ 进阶　|　**预计时长**：30–40 分钟

> **实验说明**
> 1. 本实验采用分步实现的方式：由 **v1 串行版本**起，依次引入 **v2 NEON 规约**、**v3 多累加变量展开**与 **v4 寄存器分块**，每一版本均实际编译并运行，据此观察性能的逐步变化。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **ARM(aarch64/arm64) + NEON**；请在华为鲲鹏处理器上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 四个版本的代码为递增关系：后一版本在前一版本基础上新增一个实现方法，输出表格相应增加一行，便于对照阅读，理解优化的引入过程。
> 6. 本实验建立在**实验一 AXPY** 的基础之上，建议先完成实验一。

## 🎯 学习目标

完成本实验后，学生应能够：

- 理解**规约（reduction）** 运算的含义，及其相对逐元素运算更难向量化的原因
- 掌握**水平规约**指令 `vaddvq_f32`，理解“中间和保留在向量寄存器、末尾一次性归约为标量”的思想
- 区分**延迟（latency）与吞吐（throughput）**，理解 **RAW 数据依赖**对流水线的影响
- 掌握**多累加变量**技术，理解其通过打破依赖链提高指令级并行（ILP）的原理
- 掌握**寄存器分块**（M4：一次处理 4 行），理解其通过复用向量寄存器提高**算术强度**的机制
- 观察并解释一个重要现象：**最优实现随矩阵规模变化**，以及**自动向量化对规约运算的局限性**

## 🗺️ 学习路径

1. **准备阶段**：理解 GEMV 的定义及其与 AXPY 的区别（引入了“求和”）
2. **v1 · 串行基准**：实现 `matvec_serial_no_vec`（关闭向量化，作为基准）与 `matvec_serial`（允许自动向量化）
   → 考察编译器对**规约型循环**的自动向量化能力
3. **v2 · NEON 规约**：新增 `matvec_neon`，采用向量累加 + 水平规约
   → 掌握规约的向量化实现
4. **v3 · 多累加变量**：新增 `matvec_neon_unroll_n16`，用 4 个独立累加器打破依赖链
   → 考察打破依赖对性能的影响
5. **v4 · 寄存器分块**：新增 `matvec_neon_unroll_n16_m4`，一次处理 4 行以复用 x
   → 考察提高数据复用（算术强度）的效果
6. **可视化与分析**：以 v4 的输出结果绘制加速比柱状图，分析规模相关性与规约的向量化难点

## 1. 背景与动机

GEMV 是 BLAS **Level-2** 操作：矩阵与向量相乘，`y = A · x`。其计算量为 O(n²)，高于 AXPY 的 O(n)，算术强度也相应提高，为向量化提供了更大的优化空间。

但 GEMV 引入了一个 AXPY 所没有的难点：`y[i]` 是**矩阵第 i 行与向量 x 的点积**，需要将若干乘积**累加为一个标量**。这种“多对一”的**规约（reduction）** 运算，是 SIMD 编程遇到的第一个实质性难点——它使得向量各通道之间产生了依赖关系，无法像 AXPY 那样简单地逐通道并行。

## 2. 算法与公式

$$y_i = \sum_{j=0}^{N-1} A_{ij}\, x_j$$

内层循环是一个**点积**：将 `A[i][j] * x[j]` 逐项累加。向量化时，先用 `vfmaq_f32` 并行计算 4 路乘加，得到 4 个**部分和**（保存在一个向量寄存器的 4 条通道中）；在一行处理完毕后，再通过**水平规约**将这 4 个部分和相加为 1 个标量。

> 关键在于：**水平规约仅在每行结束时执行一次**，内层循环全程让部分和保留在向量寄存器中累加，以避免频繁的向量-标量转换开销。

## 3. 核心 NEON 指令与技巧

```c
float32x4_t vsum = vdupq_n_f32(0.0f);        // 累加器初始化为 0
for (; j <= N - 4; j += 4) {
  float32x4_t va = vld1q_f32(row + j);       // 加载矩阵一行的 4 个元素
  float32x4_t vx = vld1q_f32(x + j);         // 加载向量 x 的 4 个元素
  vsum = vfmaq_f32(vsum, va, vx);            // 乘加，累加到 4 条通道
}
float sum = vaddvq_f32(vsum);                // 水平规约：4 通道求和为标量
for (; j < N; j++) sum += row[j] * x[j];     // 尾部以标量处理
```

- **`vfmaq_f32`**：乘加融合，将部分积累加到 4 条通道
- **`vaddvq_f32`**：**水平规约**，将一个向量的 4 条通道求和为一个标量（AArch64 指令；ARMv7 需用 `vget_low_f32`/`vget_high_f32` 相加折叠成 2 条通道，再用 `vget_lane_f32` 取出相加，代码中已给出兼容分支）
- **多累加变量**：使用多个相互独立的累加器，使各条 `vfmaq` 指令互不依赖，从而填充流水线（详见 v3）
- **寄存器分块**：一次处理多行，使加载的 x 被复用多次，提高算术强度（详见 v4）

## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
IS_ARM = platform.machine().lower() in ("aarch64", "arm64", "armv7l", "armv8l")
if not IS_ARM:
    print("\n⚠️  当前不是 ARM 架构，NEON 代码无法在此编译运行。")
elif CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
else:
    print("\n✅ 环境就绪：ARM 架构 + 编译器可用，可以开始实验！")

In [ ]:
import subprocess, platform, re, shutil

MACHINE = platform.machine().lower()


def compile_c(src, out):
    """尝试多组编译参数，返回可执行文件名；失败则打印错误。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    if MACHINE in ("armv7l", "armv8l"):  # 32 位 ARM 需显式开 NEON
        flagsets = ["-O3 -fPIC -mfpu=neon -mfloat-abi=hard -march=armv7-a"]
    else:  # aarch64 / arm64：NEON 默认开启
        flagsets = ["-O3 -fPIC"]
    last = ""
    for fl in flagsets:
        cmd = f"{base} {fl} {src} -o {out} -lm"
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            print("✅ 编译成功：", cmd)
            return out
        last = r.stderr
    print("❌ 编译失败：\n", last)
    return None


def run_bin(out, *args):
    """运行可执行文件并打印其输出。"""
    r = subprocess.run(
        [f"./{out}"] + [str(a) for a in args], capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout


def parse_table(text):
    """解析 | 方法 | 耗时 | 加速比 | 校验 | 表格，兼容 '4.21 x' 与 '4.21x'。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 3:
            continue
        name = cells[0]
        if name.lower() in ("method", "方法") or set(name) <= set("-: "):
            continue
        mt = re.search(r"[-+]?\d*\.?\d+", cells[1])
        ms = re.search(r"[-+]?\d*\.?\d+", cells[2])
        if not mt:
            continue
        rows.append(
            {
                "method": name,
                "time": float(mt.group()),
                "speedup": float(ms.group()) if ms else None,
            }
        )
    return rows


In [ ]:
import matplotlib.pyplot as plt


def plot_speedup(rows, title=""):
    rows = [r for r in rows if r["speedup"] is not None]
    if not rows:
        print("未解析到可绘制的加速比。")
        return
    names = [r["method"] for r in rows]
    sp = [r["speedup"] for r in rows]
    best = sp.index(max(sp))
    colors = ["#9aa0a6" if s <= 1.05 else "#295E96" for s in sp]
    colors[best] = "#C7000B"  # 最快版本标红
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, sp, color=colors)
    plt.axhline(1.0, ls="--", c="gray", lw=1)
    for b, s in zip(bars, sp):
        plt.text(
            b.get_x() + b.get_width() / 2,
            s,
            f"{s:.2f}x",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    plt.ylabel("Speedup (x)")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# 创建源代码目录
!mkdir -p src_gemv

## 5. v1 · 串行基准实现

与实验一相同，第一个版本包含**两个函数体相同**的串行实现，区别仅在于是否允许编译器自动向量化：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">函数</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>matvec_serial_no_vec</code></td>
      <td style="text-align: left;">通过 <code>no-tree-vectorize</code> <strong>显式关闭</strong>自动向量化，作为<strong>性能基准</strong>（1.00×）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>matvec_serial</code></td>
      <td style="text-align: left;">源码相同，但<strong>允许编译器自动向量化</strong></td>
    </tr>
  </tbody>
</table>

### 💡 关注点：编译器能否自动向量化规约循环
AXPY 是逐元素运算，编译器容易自动向量化；而 GEMV 的内层是**规约**（点积），各次累加之间存在数据依赖。请在运行后观察 `Serial (Auto)` 相对基准的加速比——它是否也像 AXPY 那样获得了明显加速？这一对比将在后续分析中揭示规约运算的特殊性。

### 数据布局
- `A`：M×N 矩阵，按行主序（row-major）存储；`x`：长度 N 的向量；`y`：长度 M 的结果向量
- 与 AXPY 不同，GEMV 的结果 `y` 是**新写入**的（非原地更新）
- 参考结果 `y_ref` 由 `matvec_serial_no_vec` 生成；`check_result` 容差取 `1e-4`（略高于 AXPY，因为向量化改变了累加顺序，浮点舍入误差会有微小差异）

In [ ]:
%%writefile src_gemv/gemv_v1.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

const char* check_result(const float* restrict ref, const float* restrict test,
                         int M) {
  double max_diff = 0.0;
  for (int i = 0; i < M; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
  }
  // Tolerance slightly higher due to accumulation order differences in
  // reduction
  if (max_diff < 1e-4) return "PASS";
  return "FAIL";
}

// ---------------------------------------------------------
// 1. Serial Version (Compiler may auto-vectorize)
// ---------------------------------------------------------
void matvec_serial(const float* restrict A, const float* restrict x,
                   float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    float sum = 0.0f;
    const float* restrict row = A + (long)i * N;
    for (int j = 0; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ---------------------------------------------------------
// 2. Serial Version (Forced No-Vectorization) - Baseline
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void matvec_serial_no_vec(const float* restrict A, const float* restrict x,
                          float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    float sum = 0.0f;
    const float* restrict row = A + (long)i * N;
    for (int j = 0; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

int main(int argc, char** argv) {
  if (argc != 3) {
    printf("Usage: %s <M_rows> <N_cols>\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  if (M <= 0 || N <= 0) return 1;

  printf("============================================================\n");
  printf(" GEMV v1: Serial Baseline and Auto-Vectorized Serial (y = A * x)\n");
  printf(" Matrix: %d x %d\n", M, N);
  printf(" Loops:  %d\n", NTIMES);
  printf("============================================================\n");

  // Aligned allocation (16-byte) for SIMD efficiency
  // size must be a multiple of the alignment (C11 requirement)
  size_t size_A = ((size_t)M * N * sizeof(float) + 15) & ~(size_t)15;
  size_t size_x = ((size_t)N * sizeof(float) + 15) & ~(size_t)15;
  size_t size_y = ((size_t)M * sizeof(float) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, size_A);
  float* x = (float*)aligned_alloc(16, size_x);
  float* y_ref = (float*)aligned_alloc(16, size_y);   // Golden result
  float* y_test = (float*)aligned_alloc(16, size_y);  // Reusable buffer

  if (!A || !x || !y_ref || !y_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      A[(long)i * N + j] = (float)((i + j) % 100) * 0.001f;
    }
  }
  for (int j = 0; j < N; j++) x[j] = (float)(j % 100) * 0.001f;

  // Golden reference = Serial (No-Vec)
  matvec_serial_no_vec(A, x, y_ref, M, N);

  double start, end;

  // Baseline: Serial (No-Vec)
  memset(y_test, 0, size_y);
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_serial_no_vec(A, x, y_test, M, N);
  end = get_time_ms();
  double t_no_vec = (end - start) / NTIMES;
  const char* s_no_vec = check_result(y_ref, y_test, M);

  // Serial (Auto-Vec)
  memset(y_test, 0, size_y);
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_serial(A, x, y_test, M, N);
  end = get_time_ms();
  double t_serial = (end - start) / NTIMES;
  const char* s_serial = check_result(y_ref, y_test, M);

  // Report
  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |  %-4s |\n", t_no_vec, s_no_vec);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_serial,
         t_no_vec / t_serial, s_serial);
  printf("-------------------------------------------------\n");

  free(A);
  free(x);
  free(y_ref);
  free(y_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemv/gemv_v1.c", "src_gemv/gemv_v1")
out_v1 = run_bin(BIN, 4096, 4096)

## 6. v2 · NEON 规约实现

在 v1 的基础上**新增 `matvec_neon` 函数**，实现规约的向量化：

```c
float32x4_t vsum = vdupq_n_f32(0.0f);
for (; j <= N - 4; j += 4) {
  vsum = vfmaq_f32(vsum, vld1q_f32(row + j), vld1q_f32(x + j));  // 4 路部分和
}
float sum = vaddvq_f32(vsum);       // 水平规约为标量
for (; j < N; j++) sum += row[j] * x[j];   // 尾部标量处理
```

### 🔑 知识点
- **部分和保留在向量寄存器中**：内层循环全程用 `vsum` 累加，不逐步转换为标量，避免频繁的向量-标量传输
- **水平规约只做一次**：`vaddvq_f32` 在每行末尾将 4 条通道合并为一个标量
- **规约与逐元素的差异**：AXPY 的输出与输入一一对应，GEMV 的输出是多个乘积之**和**，这正是需要“水平规约”这一额外步骤的原因

此版本仅使用**单个累加器**，各次 `vfmaq` 之间存在依赖，尚未充分利用流水线，v3 将对此改进。

In [ ]:
%%writefile src_gemv/gemv_v2.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

const char* check_result(const float* restrict ref, const float* restrict test,
                         int M) {
  double max_diff = 0.0;
  for (int i = 0; i < M; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
  }
  // Tolerance slightly higher due to accumulation order differences in
  // reduction
  if (max_diff < 1e-4) return "PASS";
  return "FAIL";
}

// ---------------------------------------------------------
// 1. Serial Version (Compiler may auto-vectorize)
// ---------------------------------------------------------
void matvec_serial(const float* restrict A, const float* restrict x,
                   float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    float sum = 0.0f;
    const float* restrict row = A + (long)i * N;
    for (int j = 0; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ---------------------------------------------------------
// 2. Serial Version (Forced No-Vectorization) - Baseline
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void matvec_serial_no_vec(const float* restrict A, const float* restrict x, float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    float sum = 0.0f;
    const float* restrict row = A + (long)i * N;
    for (int j = 0; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ---------------------------------------------------------
// 3. NEON Version (Basic reduction with horizontal add)
// ---------------------------------------------------------
void matvec_neon(const float* restrict A, const float* restrict x,
                 float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    const float* row = A + (long)i * N;
    int j = 0;

    // Initialize accumulator with 0
    float32x4_t vsum = vdupq_n_f32(0.0f);

    for (; j <= N - 4; j += 4) {
      float32x4_t va = vld1q_f32(row + j);
      float32x4_t vx = vld1q_f32(x + j);
      vsum = vfmaq_f32(vsum, va, vx);  // Fused Multiply-Add
    }

    // Horizontal reduction: sum all 4 lanes into a scalar
#if defined(__aarch64__)
    // ARMv8 instruction: efficient add across vector
    float sum = vaddvq_f32(vsum);
#else
    // Fallback for older ARMv7
    float32x2_t vlow = vget_low_f32(vsum);
    float32x2_t vhigh = vget_high_f32(vsum);
    float32x2_t vpair = vadd_f32(vlow, vhigh);
    float sum = vget_lane_f32(vpair, 0) + vget_lane_f32(vpair, 1);
#endif

    // Clean up tail
    for (; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

int main(int argc, char** argv) {
  if (argc != 3) {
    printf("Usage: %s <M_rows> <N_cols>\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  if (M <= 0 || N <= 0) return 1;

  printf("============================================================\n");
  printf(" GEMV v2: Add NEON Basic Reduction (y = A * x)\n");
  printf(" Matrix: %d x %d\n", M, N);
  printf(" Loops:  %d\n", NTIMES);
  printf("============================================================\n");

  // Aligned allocation (16-byte) for SIMD efficiency
  // size must be a multiple of the alignment (C11 requirement)
  size_t size_A = ((size_t)M * N * sizeof(float) + 15) & ~(size_t)15;
  size_t size_x = ((size_t)N * sizeof(float) + 15) & ~(size_t)15;
  size_t size_y = ((size_t)M * sizeof(float) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, size_A);
  float* x = (float*)aligned_alloc(16, size_x);
  float* y_ref = (float*)aligned_alloc(16, size_y);   // Golden result
  float* y_test = (float*)aligned_alloc(16, size_y);  // Reusable buffer

  if (!A || !x || !y_ref || !y_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      A[(long)i * N + j] = (float)((i + j) % 100) * 0.001f;
    }
  }
  for (int j = 0; j < N; j++) x[j] = (float)(j % 100) * 0.001f;

  // Golden reference = Serial (No-Vec)
  matvec_serial_no_vec(A, x, y_ref, M, N);

  double start, end;

  // Baseline: Serial (No-Vec)
  memset(y_test, 0, size_y);
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_serial_no_vec(A, x, y_test, M, N);
  end = get_time_ms();
  double t_no_vec = (end - start) / NTIMES;
  const char* s_no_vec = check_result(y_ref, y_test, M);

  // Serial (Auto-Vec)
  memset(y_test, 0, size_y);
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_serial(A, x, y_test, M, N);
  end = get_time_ms();
  double t_serial = (end - start) / NTIMES;
  const char* s_serial = check_result(y_ref, y_test, M);

  // NEON Basic
  memset(y_test, 0, size_y);
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_neon(A, x, y_test, M, N);
  end = get_time_ms();
  double t_neon = (end - start) / NTIMES;
  const char* s_neon = check_result(y_ref, y_test, M);

  // Report
  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |  %-4s |\n", t_no_vec, s_no_vec);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_serial,
         t_no_vec / t_serial, s_serial);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_neon,
         t_no_vec / t_neon, s_neon);
  printf("-------------------------------------------------\n");

  free(A);
  free(x);
  free(y_ref);
  free(y_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemv/gemv_v2.c", "src_gemv/gemv_v2")
out_v2 = run_bin(BIN, 4096, 4096)

## 7. v3 · 多累加变量（打破依赖链）

在 v2 的基础上**新增 `matvec_neon_unroll_n16` 函数**：使用 **4 个相互独立的累加器**，单次迭代处理 16 个元素。

```c
float32x4_t vsum0=0, vsum1=0, vsum2=0, vsum3=0;   // 4 个独立累加器
for (; j <= N - 16; j += 16) {
  vsum0 = vfmaq_f32(vsum0, vld1q_f32(row+j),    vld1q_f32(x+j));
  vsum1 = vfmaq_f32(vsum1, vld1q_f32(row+j+4),  vld1q_f32(x+j+4));
  vsum2 = vfmaq_f32(vsum2, vld1q_f32(row+j+8),  vld1q_f32(x+j+8));
  vsum3 = vfmaq_f32(vsum3, vld1q_f32(row+j+12), vld1q_f32(x+j+12));
}
float32x4_t vsum = vaddq_f32(vaddq_f32(vsum0,vsum1), vaddq_f32(vsum2,vsum3));
```

### 🔑 知识点：为何单累加器会限制性能
浮点乘加指令 `vfmaq` 具有若干周期的**执行延迟**。若只用一个累加器，则第 k+1 次累加必须等待第 k 次的结果写回后才能开始（**RAW 数据依赖**），处理器只能串行等待，无法填满流水线。

使用**多个相互独立的累加器**后，各条 `vfmaq` 指令之间不存在依赖，处理器可将它们在流水线中重叠执行，从而**隐藏 FMA 的执行延迟**、提高指令级并行度。所需独立累加器的数量约等于 **指令延迟 × 发射宽度**。

> 该思想与实验一 v3 的“多个独立向量变量”一脉相承，但在此处，由于内层是规约、存在真实的依赖链，多累加变量的作用更为关键。

In [ ]:
%%writefile src_gemv/gemv_v3.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

const char* check_result(const float* restrict ref, const float* restrict test,
                         int M) {
  double max_diff = 0.0;
  for (int i = 0; i < M; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
  }
  // Tolerance slightly higher due to accumulation order differences in
  // reduction
  if (max_diff < 1e-4) return "PASS";
  return "FAIL";
}

// ---------------------------------------------------------
// 1. Serial Version (Compiler may auto-vectorize)
// ---------------------------------------------------------
void matvec_serial(const float* restrict A, const float* restrict x,
                   float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    float sum = 0.0f;
    const float* restrict row = A + (long)i * N;
    for (int j = 0; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ---------------------------------------------------------
// 2. Serial Version (Forced No-Vectorization) - Baseline
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void matvec_serial_no_vec(const float* restrict A, const float* restrict x, float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    float sum = 0.0f;
    const float* restrict row = A + (long)i * N;
    for (int j = 0; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ---------------------------------------------------------
// 3. NEON Version (Basic reduction with horizontal add)
// ---------------------------------------------------------
void matvec_neon(const float* restrict A, const float* restrict x,
                 float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    const float* row = A + (long)i * N;
    int j = 0;

    // Initialize accumulator with 0
    float32x4_t vsum = vdupq_n_f32(0.0f);

    for (; j <= N - 4; j += 4) {
      float32x4_t va = vld1q_f32(row + j);
      float32x4_t vx = vld1q_f32(x + j);
      vsum = vfmaq_f32(vsum, va, vx);  // Fused Multiply-Add
    }

    // Horizontal reduction: sum all 4 lanes into a scalar
#if defined(__aarch64__)
    // ARMv8 instruction: efficient add across vector
    float sum = vaddvq_f32(vsum);
#else
    // Fallback for older ARMv7
    float32x2_t vlow = vget_low_f32(vsum);
    float32x2_t vhigh = vget_high_f32(vsum);
    float32x2_t vpair = vadd_f32(vlow, vhigh);
    float sum = vget_lane_f32(vpair, 0) + vget_lane_f32(vpair, 1);
#endif

    // Clean up tail
    for (; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ---------------------------------------------------------
// 4. NEON Version (Unroll N16, 4 independent accumulators)
// ---------------------------------------------------------
void matvec_neon_unroll_n16(const float* restrict A, const float* restrict x,
                            float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    const float* row = A + (long)i * N;
    int j = 0;

    // Use 4 independent accumulators to break dependency chains
    float32x4_t vsum0 = vdupq_n_f32(0.0f);
    float32x4_t vsum1 = vdupq_n_f32(0.0f);
    float32x4_t vsum2 = vdupq_n_f32(0.0f);
    float32x4_t vsum3 = vdupq_n_f32(0.0f);

    for (; j <= N - 16; j += 16) {
      float32x4_t vx0 = vld1q_f32(x + j);
      float32x4_t vx1 = vld1q_f32(x + j + 4);
      float32x4_t vx2 = vld1q_f32(x + j + 8);
      float32x4_t vx3 = vld1q_f32(x + j + 12);

      float32x4_t va0 = vld1q_f32(row + j);
      float32x4_t va1 = vld1q_f32(row + j + 4);
      float32x4_t va2 = vld1q_f32(row + j + 8);
      float32x4_t va3 = vld1q_f32(row + j + 12);

      // Independent FMA operations - CPU can pipeline these!
      vsum0 = vfmaq_f32(vsum0, va0, vx0);
      vsum1 = vfmaq_f32(vsum1, va1, vx1);
      vsum2 = vfmaq_f32(vsum2, va2, vx2);
      vsum3 = vfmaq_f32(vsum3, va3, vx3);
    }

    // Sum up the 4 accumulators
    float32x4_t vsum =
        vaddq_f32(vaddq_f32(vsum0, vsum1), vaddq_f32(vsum2, vsum3));

    // Handle remaining chunks of 4
    for (; j <= N - 4; j += 4) {
      float32x4_t va = vld1q_f32(row + j);
      float32x4_t vx = vld1q_f32(x + j);
      vsum = vfmaq_f32(vsum, va, vx);
    }

    // Final reduction
#if defined(__aarch64__)
    float sum = vaddvq_f32(vsum);
#else
    float32x2_t vlow = vget_low_f32(vsum);
    float32x2_t vhigh = vget_high_f32(vsum);
    float32x2_t vpair = vadd_f32(vlow, vhigh);
    float sum = vget_lane_f32(vpair, 0) + vget_lane_f32(vpair, 1);
#endif

    // Tail
    for (; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

int main(int argc, char** argv) {
  if (argc != 3) {
    printf("Usage: %s <M_rows> <N_cols>\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  if (M <= 0 || N <= 0) return 1;

  printf("============================================================\n");
  printf(" GEMV v3: Add NEON Multi-Accumulator Unroll N16 (y = A * x)\n");
  printf(" Matrix: %d x %d\n", M, N);
  printf(" Loops:  %d\n", NTIMES);
  printf("============================================================\n");

  // Aligned allocation (16-byte) for SIMD efficiency
  // size must be a multiple of the alignment (C11 requirement)
  size_t size_A = ((size_t)M * N * sizeof(float) + 15) & ~(size_t)15;
  size_t size_x = ((size_t)N * sizeof(float) + 15) & ~(size_t)15;
  size_t size_y = ((size_t)M * sizeof(float) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, size_A);
  float* x = (float*)aligned_alloc(16, size_x);
  float* y_ref = (float*)aligned_alloc(16, size_y);   // Golden result
  float* y_test = (float*)aligned_alloc(16, size_y);  // Reusable buffer

  if (!A || !x || !y_ref || !y_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      A[(long)i * N + j] = (float)((i + j) % 100) * 0.001f;
    }
  }
  for (int j = 0; j < N; j++) x[j] = (float)(j % 100) * 0.001f;

  // Golden reference = Serial (No-Vec)
  matvec_serial_no_vec(A, x, y_ref, M, N);

  double start, end;

  // Baseline: Serial (No-Vec)
  memset(y_test, 0, size_y);
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_serial_no_vec(A, x, y_test, M, N);
  end = get_time_ms();
  double t_no_vec = (end - start) / NTIMES;
  const char* s_no_vec = check_result(y_ref, y_test, M);

  // Serial (Auto-Vec)
  memset(y_test, 0, size_y);
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_serial(A, x, y_test, M, N);
  end = get_time_ms();
  double t_serial = (end - start) / NTIMES;
  const char* s_serial = check_result(y_ref, y_test, M);

  // NEON Basic
  memset(y_test, 0, size_y);
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_neon(A, x, y_test, M, N);
  end = get_time_ms();
  double t_neon = (end - start) / NTIMES;
  const char* s_neon = check_result(y_ref, y_test, M);

  // NEON Unroll N16 (multiple accumulators)
  memset(y_test, 0, size_y);
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_neon_unroll_n16(A, x, y_test, M, N);
  end = get_time_ms();
  double t_n16 = (end - start) / NTIMES;
  const char* s_n16 = check_result(y_ref, y_test, M);

  // Report
  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |  %-4s |\n", t_no_vec, s_no_vec);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_serial,
         t_no_vec / t_serial, s_serial);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_neon,
         t_no_vec / t_neon, s_neon);
  printf("| NEON Unroll N16 | %9.3f | %5.2f x |  %-4s |\n", t_n16,
         t_no_vec / t_n16, s_n16);
  printf("-------------------------------------------------\n");

  free(A);
  free(x);
  free(y_ref);
  free(y_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemv/gemv_v3.c", "src_gemv/gemv_v3")
out_v3 = run_bin(BIN, 4096, 4096)

## 8. v4 · 寄存器分块（提高数据复用）

在 v3 的基础上**新增 `matvec_neon_unroll_n16_m4` 函数**：外层一次处理 **4 行**（M 方向分块），内层沿用 N16 的多累加器结构。

```c
for (; i <= M - 4; i += 4) {          // 一次处理 4 行
  const float *row0=..., *row1=..., *row2=..., *row3=...;
  for (; j <= N - 16; j += 16) {
    float32x4_t vx0 = vld1q_f32(x + j);   // x 加载一次
    ...
    vsum0_0 = vfmaq_f32(vsum0_0, vld1q_f32(row0+j), vx0);  // 行 0 复用 vx0
    vsum1_0 = vfmaq_f32(vsum1_0, vld1q_f32(row1+j), vx0);  // 行 1 复用 vx0
    vsum2_0 = vfmaq_f32(vsum2_0, vld1q_f32(row2+j), vx0);  // 行 2 复用 vx0
    vsum3_0 = vfmaq_f32(vsum3_0, vld1q_f32(row3+j), vx0);  // 行 3 复用 vx0
    ...
  }
}
```

### 🔑 知识点：算术强度与寄存器分块
在 v2、v3 中，向量 `x` 的每个元素在每一行都要重新加载一次。当一次处理 4 行时，加载进寄存器的 `vx0` 等可被 **4 行共用**，即**加载一次、参与 4 次乘加**。

这提高了**算术强度**（每次访存对应的计算量），使运算从“访存受限”向“计算受限”方向移动。这也是通用矩阵乘（GEMM，见第三章压轴案例）能够逼近处理器算力峰值的核心手段之一。

> 该版本共使用 16 个累加向量寄存器（4 行 × 4 个），已接近寄存器数量上限，是数据复用与寄存器压力之间的一种平衡。

至此，**五个实现**（含基准）全部就位，输出表格与完整版源码 `02_neon_matvec.c` 保持一致。

In [ ]:
%%writefile src_gemv/gemv_v4.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

const char* check_result(const float* restrict ref, const float* restrict test,
                         int M) {
  double max_diff = 0.0;
  for (int i = 0; i < M; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
  }
  // Tolerance slightly higher due to accumulation order differences in
  // reduction
  if (max_diff < 1e-4) return "PASS";
  return "FAIL";
}

// ---------------------------------------------------------
// 1. Serial Version (Compiler may auto-vectorize)
// ---------------------------------------------------------
void matvec_serial(const float* restrict A, const float* restrict x,
                   float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    float sum = 0.0f;
    const float* restrict row = A + (long)i * N;
    for (int j = 0; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ---------------------------------------------------------
// 2. Serial Version (Forced No-Vectorization) - Baseline
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void matvec_serial_no_vec(const float* restrict A, const float* restrict x, float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    float sum = 0.0f;
    const float* restrict row = A + (long)i * N;
    for (int j = 0; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ---------------------------------------------------------
// 3. NEON Version (Basic reduction with horizontal add)
// ---------------------------------------------------------
void matvec_neon(const float* restrict A, const float* restrict x,
                 float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    const float* row = A + (long)i * N;
    int j = 0;

    // Initialize accumulator with 0
    float32x4_t vsum = vdupq_n_f32(0.0f);

    for (; j <= N - 4; j += 4) {
      float32x4_t va = vld1q_f32(row + j);
      float32x4_t vx = vld1q_f32(x + j);
      vsum = vfmaq_f32(vsum, va, vx);  // Fused Multiply-Add
    }

    // Horizontal reduction: sum all 4 lanes into a scalar
#if defined(__aarch64__)
    // ARMv8 instruction: efficient add across vector
    float sum = vaddvq_f32(vsum);
#else
    // Fallback for older ARMv7
    float32x2_t vlow = vget_low_f32(vsum);
    float32x2_t vhigh = vget_high_f32(vsum);
    float32x2_t vpair = vadd_f32(vlow, vhigh);
    float sum = vget_lane_f32(vpair, 0) + vget_lane_f32(vpair, 1);
#endif

    // Clean up tail
    for (; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ---------------------------------------------------------
// 4. NEON Version (Unroll N16, 4 independent accumulators)
// ---------------------------------------------------------
void matvec_neon_unroll_n16(const float* restrict A, const float* restrict x,
                            float* restrict y, int M, int N) {
  for (int i = 0; i < M; i++) {
    const float* row = A + (long)i * N;
    int j = 0;

    // Use 4 independent accumulators to break dependency chains
    float32x4_t vsum0 = vdupq_n_f32(0.0f);
    float32x4_t vsum1 = vdupq_n_f32(0.0f);
    float32x4_t vsum2 = vdupq_n_f32(0.0f);
    float32x4_t vsum3 = vdupq_n_f32(0.0f);

    for (; j <= N - 16; j += 16) {
      float32x4_t vx0 = vld1q_f32(x + j);
      float32x4_t vx1 = vld1q_f32(x + j + 4);
      float32x4_t vx2 = vld1q_f32(x + j + 8);
      float32x4_t vx3 = vld1q_f32(x + j + 12);

      float32x4_t va0 = vld1q_f32(row + j);
      float32x4_t va1 = vld1q_f32(row + j + 4);
      float32x4_t va2 = vld1q_f32(row + j + 8);
      float32x4_t va3 = vld1q_f32(row + j + 12);

      // Independent FMA operations - CPU can pipeline these!
      vsum0 = vfmaq_f32(vsum0, va0, vx0);
      vsum1 = vfmaq_f32(vsum1, va1, vx1);
      vsum2 = vfmaq_f32(vsum2, va2, vx2);
      vsum3 = vfmaq_f32(vsum3, va3, vx3);
    }

    // Sum up the 4 accumulators
    float32x4_t vsum =
        vaddq_f32(vaddq_f32(vsum0, vsum1), vaddq_f32(vsum2, vsum3));

    // Handle remaining chunks of 4
    for (; j <= N - 4; j += 4) {
      float32x4_t va = vld1q_f32(row + j);
      float32x4_t vx = vld1q_f32(x + j);
      vsum = vfmaq_f32(vsum, va, vx);
    }

    // Final reduction
#if defined(__aarch64__)
    float sum = vaddvq_f32(vsum);
#else
    float32x2_t vlow = vget_low_f32(vsum);
    float32x2_t vhigh = vget_high_f32(vsum);
    float32x2_t vpair = vadd_f32(vlow, vhigh);
    float sum = vget_lane_f32(vpair, 0) + vget_lane_f32(vpair, 1);
#endif

    // Tail
    for (; j < N; j++) {
      sum += row[j] * x[j];
    }
    y[i] = sum;
  }
}

// ---------------------------------------------------------
// 5. NEON Version (Unroll N16 + Register Blocking M4)
// ---------------------------------------------------------
void matvec_neon_unroll_n16_m4(const float* restrict A, const float* restrict x,
                               float* restrict y, int M, int N) {
  int i = 0;
  // Outer loop: process 4 rows at a time to maximize register reuse
  for (; i <= M - 4; i += 4) {
    const float* row0 = A + (long)(i + 0) * N;
    const float* row1 = A + (long)(i + 1) * N;
    const float* row2 = A + (long)(i + 2) * N;
    const float* row3 = A + (long)(i + 3) * N;

    // Use 4 independent accumulators per row to maintain original logic (16
    // registers total)
    float32x4_t vsum0_0 = vdupq_n_f32(0.0f), vsum0_1 = vdupq_n_f32(0.0f),
                vsum0_2 = vdupq_n_f32(0.0f), vsum0_3 = vdupq_n_f32(0.0f);
    float32x4_t vsum1_0 = vdupq_n_f32(0.0f), vsum1_1 = vdupq_n_f32(0.0f),
                vsum1_2 = vdupq_n_f32(0.0f), vsum1_3 = vdupq_n_f32(0.0f);
    float32x4_t vsum2_0 = vdupq_n_f32(0.0f), vsum2_1 = vdupq_n_f32(0.0f),
                vsum2_2 = vdupq_n_f32(0.0f), vsum2_3 = vdupq_n_f32(0.0f);
    float32x4_t vsum3_0 = vdupq_n_f32(0.0f), vsum3_1 = vdupq_n_f32(0.0f),
                vsum3_2 = vdupq_n_f32(0.0f), vsum3_3 = vdupq_n_f32(0.0f);

    int j = 0;
    // Inner loop: process 16 elements (4 vectors) in N dimension
    for (; j <= N - 16; j += 16) {
      // Load 4 vectors of x once and reuse them for all 4 rows
      float32x4_t vx0 = vld1q_f32(x + j);
      float32x4_t vx1 = vld1q_f32(x + j + 4);
      float32x4_t vx2 = vld1q_f32(x + j + 8);
      float32x4_t vx3 = vld1q_f32(x + j + 12);

      // Row 0 FMAs
      vsum0_0 = vfmaq_f32(vsum0_0, vld1q_f32(row0 + j), vx0);
      vsum0_1 = vfmaq_f32(vsum0_1, vld1q_f32(row0 + j + 4), vx1);
      vsum0_2 = vfmaq_f32(vsum0_2, vld1q_f32(row0 + j + 8), vx2);
      vsum0_3 = vfmaq_f32(vsum0_3, vld1q_f32(row0 + j + 12), vx3);

      // Row 1 FMAs
      vsum1_0 = vfmaq_f32(vsum1_0, vld1q_f32(row1 + j), vx0);
      vsum1_1 = vfmaq_f32(vsum1_1, vld1q_f32(row1 + j + 4), vx1);
      vsum1_2 = vfmaq_f32(vsum1_2, vld1q_f32(row1 + j + 8), vx2);
      vsum1_3 = vfmaq_f32(vsum1_3, vld1q_f32(row1 + j + 12), vx3);

      // Row 2 FMAs
      vsum2_0 = vfmaq_f32(vsum2_0, vld1q_f32(row2 + j), vx0);
      vsum2_1 = vfmaq_f32(vsum2_1, vld1q_f32(row2 + j + 4), vx1);
      vsum2_2 = vfmaq_f32(vsum2_2, vld1q_f32(row2 + j + 8), vx2);
      vsum2_3 = vfmaq_f32(vsum2_3, vld1q_f32(row2 + j + 12), vx3);

      // Row 3 FMAs
      vsum3_0 = vfmaq_f32(vsum3_0, vld1q_f32(row3 + j), vx0);
      vsum3_1 = vfmaq_f32(vsum3_1, vld1q_f32(row3 + j + 4), vx1);
      vsum3_2 = vfmaq_f32(vsum3_2, vld1q_f32(row3 + j + 8), vx2);
      vsum3_3 = vfmaq_f32(vsum3_3, vld1q_f32(row3 + j + 12), vx3);
    }

    // Combine 4 accumulators per row into one vsum per row
    float32x4_t vsum0 =
        vaddq_f32(vaddq_f32(vsum0_0, vsum0_1), vaddq_f32(vsum0_2, vsum0_3));
    float32x4_t vsum1 =
        vaddq_f32(vaddq_f32(vsum1_0, vsum1_1), vaddq_f32(vsum1_2, vsum1_3));
    float32x4_t vsum2 =
        vaddq_f32(vaddq_f32(vsum2_0, vsum2_1), vaddq_f32(vsum2_2, vsum2_3));
    float32x4_t vsum3 =
        vaddq_f32(vaddq_f32(vsum3_0, vsum3_1), vaddq_f32(vsum3_2, vsum3_3));

    // Handle remaining chunks of 4 in N dimension
    for (; j <= N - 4; j += 4) {
      float32x4_t vx = vld1q_f32(x + j);
      vsum0 = vfmaq_f32(vsum0, vld1q_f32(row0 + j), vx);
      vsum1 = vfmaq_f32(vsum1, vld1q_f32(row1 + j), vx);
      vsum2 = vfmaq_f32(vsum2, vld1q_f32(row2 + j), vx);
      vsum3 = vfmaq_f32(vsum3, vld1q_f32(row3 + j), vx);
    }

    // Final horizontal reduction for 4 rows
    float s0, s1, s2, s3;
#if defined(__aarch64__)
    s0 = vaddvq_f32(vsum0);
    s1 = vaddvq_f32(vsum1);
    s2 = vaddvq_f32(vsum2);
    s3 = vaddvq_f32(vsum3);
#else
    // Fallback for ARMv7
    float32x2_t p0 = vadd_f32(vget_low_f32(vsum0), vget_high_f32(vsum0));
    s0 = vget_lane_f32(p0, 0) + vget_lane_f32(p0, 1);
    float32x2_t p1 = vadd_f32(vget_low_f32(vsum1), vget_high_f32(vsum1));
    s1 = vget_lane_f32(p1, 0) + vget_lane_f32(p1, 1);
    float32x2_t p2 = vadd_f32(vget_low_f32(vsum2), vget_high_f32(vsum2));
    s2 = vget_lane_f32(p2, 0) + vget_lane_f32(p2, 1);
    float32x2_t p3 = vadd_f32(vget_low_f32(vsum3), vget_high_f32(vsum3));
    s3 = vget_lane_f32(p3, 0) + vget_lane_f32(p3, 1);
#endif

    // Scalar tail for each of the 4 rows
    for (int k = j; k < N; k++) {
      float xk = x[k];
      s0 += row0[k] * xk;
      s1 += row1[k] * xk;
      s2 += row2[k] * xk;
      s3 += row3[k] * xk;
    }
    y[i + 0] = s0;
    y[i + 1] = s1;
    y[i + 2] = s2;
    y[i + 3] = s3;
  }

  // Handle remaining rows in M dimension
  for (; i < M; i++) {
    const float* row = A + (long)i * N;
    float32x4_t vsum = vdupq_n_f32(0.0f);
    int j = 0;
    for (; j <= N - 4; j += 4) {
      vsum = vfmaq_f32(vsum, vld1q_f32(row + j), vld1q_f32(x + j));
    }
    float sum = 0;
#if defined(__aarch64__)
    sum = vaddvq_f32(vsum);
#else
    float32x2_t p = vadd_f32(vget_low_f32(vsum), vget_high_f32(vsum));
    sum = vget_lane_f32(p, 0) + vget_lane_f32(p, 1);
#endif
    for (; j < N; j++) sum += row[j] * x[j];
    y[i] = sum;
  }
}

int main(int argc, char** argv) {
  if (argc != 3) {
    printf("Usage: %s <M_rows> <N_cols>\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  if (M <= 0 || N <= 0) return 1;

  printf("============================================================\n");
  printf(" GEMV v4: Add NEON Register-Blocking M4 (y = A * x)\n");
  printf(" Matrix: %d x %d\n", M, N);
  printf(" Loops:  %d\n", NTIMES);
  printf("============================================================\n");

  // Aligned allocation (16-byte) for SIMD efficiency
  // size must be a multiple of the alignment (C11 requirement)
  size_t size_A = ((size_t)M * N * sizeof(float) + 15) & ~(size_t)15;
  size_t size_x = ((size_t)N * sizeof(float) + 15) & ~(size_t)15;
  size_t size_y = ((size_t)M * sizeof(float) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, size_A);
  float* x = (float*)aligned_alloc(16, size_x);
  float* y_ref = (float*)aligned_alloc(16, size_y);   // Golden result
  float* y_test = (float*)aligned_alloc(16, size_y);  // Reusable buffer

  if (!A || !x || !y_ref || !y_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      A[(long)i * N + j] = (float)((i + j) % 100) * 0.001f;
    }
  }
  for (int j = 0; j < N; j++) x[j] = (float)(j % 100) * 0.001f;

  // Golden reference = Serial (No-Vec)
  matvec_serial_no_vec(A, x, y_ref, M, N);

  double start, end;

  // Baseline: Serial (No-Vec)
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_serial_no_vec(A, x, y_test, M, N);
  end = get_time_ms();
  double t_no_vec = (end - start) / NTIMES;
  const char* s_no_vec = check_result(y_ref, y_test, M);

  // Serial (Auto-Vec)
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_serial(A, x, y_test, M, N);
  end = get_time_ms();
  double t_serial = (end - start) / NTIMES;
  const char* s_serial = check_result(y_ref, y_test, M);

  // NEON Basic
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_neon(A, x, y_test, M, N);
  end = get_time_ms();
  double t_neon = (end - start) / NTIMES;
  const char* s_neon = check_result(y_ref, y_test, M);

  // NEON Unroll N16 (multiple accumulators)
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_neon_unroll_n16(A, x, y_test, M, N);
  end = get_time_ms();
  double t_n16 = (end - start) / NTIMES;
  const char* s_n16 = check_result(y_ref, y_test, M);

  // NEON Unroll N16 + Register Blocking M4
  start = get_time_ms();
  for (int k = 0; k < NTIMES; k++) matvec_neon_unroll_n16_m4(A, x, y_test, M, N);
  end = get_time_ms();
  double t_m4 = (end - start) / NTIMES;
  const char* s_m4 = check_result(y_ref, y_test, M);

  // Report
  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |  %-4s |\n", t_no_vec, s_no_vec);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_serial,
         t_no_vec / t_serial, s_serial);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_neon,
         t_no_vec / t_neon, s_neon);
  printf("| NEON Unroll N16 | %9.3f | %5.2f x |  %-4s |\n", t_n16,
         t_no_vec / t_n16, s_n16);
  printf("| NEON Unroll M4  | %9.3f | %5.2f x |  %-4s |\n", t_m4,
         t_no_vec / t_m4, s_m4);
  printf("-------------------------------------------------\n");

  free(A);
  free(x);
  free(y_ref);
  free(y_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemv/gemv_v4.c", "src_gemv/gemv_v4")
out_v4 = run_bin(BIN, 4096, 4096)

## 9. 📈 性能可视化（基于 v4 的五版本结果）

v4 的输出包含全部五个实现的耗时与加速比，据此绘制柱状图，以完整呈现各优化手段的效果。
（灰色表示无明显加速，蓝色表示存在加速，**红色标示性能最优的版本**）

In [ ]:
rows_v4 = parse_table(out_v4)
for r in rows_v4:
    print(f'{r["method"]:16s} {r["time"]:9.3f} ms   {r["speedup"]:.2f}x')
plot_speedup(rows_v4, "GEMV: performance of five implementations (4096x4096)")

## 10. 结果分析

> 注：具体数值随硬件平台、矩阵规模、编译器版本与系统负载而变化，请以本机实际运行结果为准；下述分析针对数据所反映的**趋势与规律**。

GEMV 的结果通常揭示以下几个规律：

**① 手写 NEON 相对基准有大幅加速。**

由于 GEMV 的算术强度高于 AXPY，向量化的收益更为可观，`NEON Intrinsic` 通常可达 3–4× 或更高，多累加与分块版本则进一步提升。

**② `Serial (Auto)` 的性能并没有提高，甚至还比基准稍慢。**

这是 GEMV 与 AXPY 的关键差异，但**原因不是"编译器没有向量化"**——恰恰相反，`gcc -O3 -fopt-info-vec` 会明确报告这个循环"loop vectorized"。真正发生的事情要看汇编：

```asm
.L19:                              // matvec_serial 的内层循环
  ldr   q2, [x6, x5]               // A 的 4 个元素
  ldr   q0, [x1, x5]               // x 的 4 个元素
  fmul  v0.4s, v0.4s, v2.4s        // ✅ 乘法向量化了，4 路并行
  dup   s4, v0.s[0]                // ❌ 又把 4 条通道逐个拆回标量
  dup   s3, v0.s[1]
  dup   s2, v0.s[2]
  dup   s0, v0.s[3]
  fadd  s1, s1, s4                 // ❌ 加法仍按原顺序串行累加
  fadd  s1, s1, s3
  fadd  s1, s2, s1
  fadd  s1, s1, s0
  bne   .L19
```

编译器**只敢向量化乘法**：浮点加法不满足结合律，改变累加顺序会改变结果，所以它必须把 4 个乘积拆回标量、按原顺序一个一个加。依赖链的长度和不向量化时完全一样（每 4 个元素 4 次串行 FADD），却多出 4 条 `dup` 的拆包开销——**这就是它有时候比基准还慢的原因**。

想确认这个解释，只要加上 `-ffast-math` 让编译器可以重排累加顺序，同一份源码立刻变成：

```asm
.L19:
  ldr   q2, [x1, x5]
  ldr   q1, [x6, x5]
  fmla  v0.4s, v2.4s, v1.4s        // 与手写 NEON 完全相同
  bne   .L19
                                   // 循环末尾用 faddp/faddv 做水平规约
```

即 `-ffast-math` 下编译器生成的就是我们手写的那份代码。**挡住编译器的从来不是"规约"这个形式，而是浮点加法的结合律。**

**③ 多累加变量（v3）相对单累加器（v2）有进一步提升。**

单累加器版本受 RAW 依赖制约，未能填满流水线；多累加变量打破了依赖链，隐藏了 FMA 延迟，因而更快。

**④ 最优实现随矩阵规模而变化。**

在**较小规模**（数据可大量驻留缓存）时，性能主要受计算与流水线制约，多累加的 **N16** 往往最快；在**较大规模**（远超缓存）时，性能转为受访存带宽制约，通过复用 x 提高算术强度的 **M4** 更具优势，常反超 N16。

---

### 🎓 结论
GEMV 揭示了 SIMD 编程的第二个核心问题：**规约、流水线依赖与数据复用**。其核心结论为——**打破依赖、隐藏延迟属于“计算侧”优化，存在上限；当规模增大、访存成为瓶颈后，提高数据复用（算术强度）才是进一步提升性能的关键** 。

## 11. 🔧 动手练习

请修改代码、重新编译并运行，观察性能的变化（建议先独立完成，再阅读思考题）：

1. 分别以 `1024 1024`（可大量驻留缓存）与 `10240 10240`（远超缓存）运行 v4，记录 `NEON Unroll N16` 与 `NEON Unroll M4` 的加速比，观察二者的相对优劣如何随规模变化。
2. 修改 v3 中独立累加器的数量（由 4 组改为 2 组或 8 组），观察对性能的影响，分析“填满流水线所需的累加器数量”。
3. 将矩阵设为长条形（如 `4 1000000` 与 `1000000 4`），观察 N 方向优化与 M 方向优化各自的适用性。
4. 【进阶】给编译参数加上 `-ffast-math` 重新编译代码，再看 `Serial (Auto)` 这一行的性能。
5. 【进阶】为 `compile_c` 的编译参数添加 `-march=native` 后重新运行，观察 `Serial (Auto)` 是否有所改善，以及是否仍明显低于手写 NEON。

## 12. 🤔 思考题

- 单累加器为何会产生流水线停顿？多累加器为何能够避免？所需的累加器数量与什么因素有关？
- 在较小与较大两种矩阵规模下，为何最优实现会由 N16 变为 M4？各自受何种因素制约？
- 寄存器分块（M4）是如何提高算术强度的？它复用了哪些数据？
- 为何 `check_result` 的容差（`1e-4`）要高于实验一 AXPY 的（`1e-5`）？

## 13. 小结与后续

本实验完成了 GEMV 从串行到高度优化的完整过程：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">新增内容</th>
      <th style="text-align: left;">涉及知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>v1</strong></td>
      <td style="text-align: left;"><code>matvec_serial_no_vec</code> + <code>matvec_serial</code></td>
      <td style="text-align: left;">性能基准、自动向量化对规约的局限</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v2</strong></td>
      <td style="text-align: left;"><code>matvec_neon</code></td>
      <td style="text-align: left;">向量累加、水平规约 <code>vaddvq_f32</code></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v3</strong></td>
      <td style="text-align: left;"><code>matvec_neon_unroll_n16</code></td>
      <td style="text-align: left;">多累加变量、RAW 依赖、指令级并行</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v4</strong></td>
      <td style="text-align: left;"><code>matvec_neon_unroll_n16_m4</code></td>
      <td style="text-align: left;">寄存器分块、数据复用、算术强度</td>
    </tr>
  </tbody>
</table>

通过 GEMV，我们认识了**规约的向量化**、**流水线依赖与多累加变量**，以及**数据复用**对突破访存瓶颈的作用。需要强调的是：**打破依赖、隐藏延迟属于计算侧优化，存在上限；提高算术强度才能进一步逼近算力屋顶。**

➡️ **后续内容：实验三 RGB→BGR**。我们将进入图像处理领域，先认识一类最“纯粹”的负载——**只有访存、没有计算**，并掌握**结构化访存指令 `vld3`/`vst3`**。